Filtrar Caminhões Mercedes

In [ ]:
import pandas as pd

df_base_Mercedes_Caminhões = pd.read_excel(r"C:\Users\gabriel.vinicius\Documents\Vscode\Web Scraping\dados_mercedes_caminhoes.xlsx")

# Encontrar duplicadas verifica todas as linhas e só consta como duplicata se todas forem iguais
df_Duplicatas = df_base_Mercedes_Caminhões.drop_duplicates()

# Limpar a coluna Modelo
def limpar_modelo(texto):
    if isinstance(texto, str):
        texto = texto.replace("Mercedes-Benz do Brasil ", "")
        texto = texto.replace("Mercedes-Benz ", "")
    return texto.strip()

#Verificar se não a duplicatas
df_Duplicatas['Modelo'] = df_Duplicatas['Modelo'].apply(limpar_modelo)

df_Limpado = df_Duplicatas

df_Limpado.to_excel('Base_Caminhoes_Mercedes.xlsx', index=False)

Unificando Bases 

In [3]:
import pandas as pd

# Corrigido: agora um dicionário com as marcas como chave
arquivos = {
    "Volvo": r"C:\Users\gabriel.vinicius\Documents\Vscode\Ficha Tecnicas - Veículos Pesados\Volvo Base.xlsx",
    "Ford": r"C:\Users\gabriel.vinicius\Documents\Vscode\Ficha Tecnicas - Veículos Pesados\Base Ford - Demonstração.xlsx",
    "Scania": r"C:\Users\gabriel.vinicius\Documents\Vscode\Ficha Tecnicas - Veículos Pesados\Scania\Scania Base.xlsx",
    "Mercedes": r"C:\Users\gabriel.vinicius\Documents\Vscode\Ficha Tecnicas - Veículos Pesados\Mercedes\Mercedes Base.xlsx"
}

# Função para classificar cada tema 
def classificar_linha(linha):
    linha = linha.lower()
    if any(p in linha for p in ['motor', 'potência', 'torque']):
        return 'Motor'
    elif any(p in linha for p in ['transmissão', 'câmbio', 'embreagem']):
        return 'Transmissao'
    elif any(p in linha for p in ['freio', 'abs', 'ebs']):
        return 'Freios'
    elif any(p in linha for p in ['suspensão', 'mola', 'amortecedor']):
        return 'Suspensao'
    elif any(p in linha for p in ['direção', 'hidráulica', 'elétrica']):
        return 'Direcao'
    elif any(p in linha for p in ['dimensão', 'peso', 'comprimento', 'largura', 'eixo']):
        return 'Dimensoes'
    elif any(p in linha for p in ['elétrico', 'bateria', 'alternador']):
        return 'Eletrico'
    else:
        return 'Outros'

# Função para separar por tema
def separar_por_tema(df):
    temas = {'Motor': [], 'Transmissao': [], 'Freios': [], 'Suspensao': [],
             'Direcao': [], 'Dimensoes': [], 'Eletrico': [], 'Outros': []}
    for _, row in df.iterrows():
        linha_texto = ' '.join(str(cell) for cell in row if pd.notna(cell))
        tema = classificar_linha(linha_texto)
        temas[tema].append(row)
    return {tema: pd.DataFrame(linhas) for tema, linhas in temas.items() if linhas}

# Organizar e carregar dados por marca
all_data = {}
for marca, path in arquivos.items():
    excel_file = pd.ExcelFile(path)
    if marca in ['Mercedes', 'Ford']:
        df = excel_file.parse(excel_file.sheet_names[0])
        all_data[marca] = separar_por_tema(df)
    else:
        all_data[marca] = {sheet: excel_file.parse(sheet) for sheet in excel_file.sheet_names}

# Unificar todas as bases por componente
componentes_unificados = {}
for marca, abas in all_data.items():
    for nome_aba, df in abas.items():
        nome_padronizado = nome_aba.strip().capitalize()
        if nome_padronizado not in componentes_unificados:
            componentes_unificados[nome_padronizado] = []
        df = df.copy()
        df.insert(0, 'Marca', marca)
        componentes_unificados[nome_padronizado].append(df)

# Consolidar e exportar para Excel final
componentes_final = {
    componente: pd.concat(lista, ignore_index=True)
    for componente, lista in componentes_unificados.items()
}

with pd.ExcelWriter("Base_Unificada_Caminhoes.xlsx", engine='xlsxwriter') as writer:
    for componente, df in componentes_final.items():
        df.to_excel(writer, sheet_name=componente[:31], index=False)